# 00 — Intro

*A modernized walkthrough of the [2016 Qiita post on convolutions-as-tensor-inner-products](http://qiita.com/uyuutosa/items/12e87f4695bd151b1d74), the spiritual ancestor of this project.*

This notebook reproduces the 2016 narrative on the C++20 rewrite: named-axis tensors, function tensors, reference tensors, and the new \_tex authoring surface. By the end you should be able to write tensor expressions whose code reads like the math.

> The library is **educational, not production**. For production needs, use Eigen / xtensor / libtorch / Kokkos / std::linalg. See [ADR-0001](../docs/arc42/09-decisions/0001-pivot-to-educational-named-axis-dsl.md) for the rationale.

## What this notebook covers

1. Defining named-axis tensors `a_i` and `b_j`.
2. The four arithmetic operations under Einstein-style label broadcast.
3. Function tensors — tensors whose elements are callables.
4. Reference tensors — recurrence-along-an-axis tensors.
5. The `_tex` authoring surface — *the formula is the program*.
6. Where this leads (autograd, WebGPU, the full Jupyter Book site).

## Setup

The cells below assume you have installed the conda environment from `environment.yml` and selected the C++20 (xeus-cling) kernel for this notebook. The `#pragma cling` directives wire up the include path and load any required helper libraries.

In [ ]:
#pragma cling add_include_path("../include")
#include <iostream>
#include <tensor/core/axis.hpp>
#include <tensor/core/shape.hpp>
#include <tensor/core/tensor.hpp>
#include <tensor/core/format.hpp>
#include <tensor/core/dynamic_shape.hpp>
#include <tensor/core/dynamic_tensor.hpp>
#include <tensor/core/ops.hpp>
#include <tensor/core/function_tensor.hpp>
#include <tensor/core/reference_tensor.hpp>
#include <tensor/tex/tex.hpp>

using tensor::core::Axis;
using tensor::core::Shape;
using tensor::core::Tensor;
using tensor::core::FunctionTensor;
using tensor::core::ReferenceTensor;
using namespace tensor::tex::literals;

## 1 — Named-axis tensors

Every tensor in `tensor::core` carries explicit axis labels. Here we define $a_i = (1, 2, 3, 4, 5)$ and $b_j = (1, 2, 3, 4, 5)$. Even though the contents look identical, the axis labels differ — and that is what makes the algebra below work.

In [ ]:
Tensor<double, 1> a(Shape<1>{Axis{"i", 5}}, {1, 2, 3, 4, 5});
Tensor<double, 1> b(Shape<1>{Axis{"j", 5}}, {1, 2, 3, 4, 5});
std::cout << a;

In [ ]:
std::cout << b;

## 2 — Four arithmetic operations under Einstein-style broadcast

Because $a$ and $b$ carry **different** axis labels, $a + b$ is interpreted as $c_{ij} = a_i + b_j$ — a rank-2 outer-product-shaped result. Same for $-, *, /$. Compare to the original 2016 README — the values match element for element.

In [ ]:
std::cout << "a + b:\n" << (a + b);

In [ ]:
std::cout << "a - b:\n" << (a - b);
std::cout << "\na * b:\n" << (a * b);
std::cout << "\na / b:\n" << (a / b);

When the two tensors share an axis label (same name **and** same extent), the algebra collapses that axis: the result is element-wise on the shared axis, no broadcast.

$c_i = a_i + a_i$ (here we use $a$ twice on purpose) yields a rank-1 tensor.

In [ ]:
std::cout << "a + a (same label, element-wise):\n" << (a + a);

## 3 — Function tensors

A *function tensor* has callables as its elements. The 2016 example — $f_i$ where each element is the same callable $\text{func}(i, v) = v + 2 i$ — is rewritten as:

In [ ]:
auto f = FunctionTensor(Axis{"i", 5},
    [](std::size_t i, double v) {
        return v + 2.0 * static_cast<double>(i);
    });
std::cout << "a * f (element i mapped through func):\n" << (a * f);

The result is `(1, 4, 7, 10, 13)`, byte-for-byte identical to the 2016 README. The modern API replaces the original raw-pointer / `new` / `delete` plumbing with a templated callable; everything else stays in the spirit of the original.

## 4 — Reference tensors

A *reference tensor* keeps a single initial value; each element along its axis is computed by referencing the previous element through the operator that fires the recurrence.

With initial value $3$ on a 5-axis named $i$, multiplying by $3$ produces:

$$r \cdot 3 = (3 \cdot 3,\ (3 \cdot 3) \cdot 3,\ \dots) = (9, 27, 81, 243, 729)$$

i.e. iterative multiplication by $3$. This is the canonical use case for solver / optimisation iterations expressed as tensor algebra.

In [ ]:
auto r = ReferenceTensor<double>(3.0, Axis{"i", 5});
std::cout << "r * 3:\n" << (r * 3.0);

## 5 — `_tex` — *the formula is the program*

[ADR-0005](../docs/arc42/09-decisions/0005-adopt-tex-lyx-as-authoring-surface.md) calls for TeX as a first-class authoring surface. The `tensor::tex` adapter parses a LaTeX subset into a structural AST that downstream code (this notebook for now, the evaluator bridge for later notebooks) can consume.

Below: write `c_{ij} = a_i + b_j` as you would in a paper. The library round-trips it back to LaTeX losslessly.

In [ ]:
auto expr = R"(c_{ij} = a_i + b_j)"_tex;
std::cout << "Parsed AST round-trips to: " << tensor::tex::to_latex(expr) << '\n';

In [ ]:
// Implicit multiplication via juxtaposition: a_i b_i → (a_i * b_i)
auto contracted = R"(a_i b_i)"_tex;
std::cout << "juxtaposition becomes implicit multiplication:\n  "
          << tensor::tex::to_latex(contracted) << '\n';

In [ ]:
// Sums and explicit grouping survive parsing too.
auto summed = R"(\sum_i {a_i b_i})"_tex;
std::cout << "sum: " << tensor::tex::to_latex(summed) << '\n';

## 6 — Where this leads

This intro notebook covers Phase 1 of [the implementation plan](../docs/impl-plans/2026-05-10_revival-phase-1.md). Phase 2 adds **autograd** ([ADR-0007](../docs/arc42/09-decisions/0007-adopt-autograd-as-first-class-subsystem.md)) — a tape-based reverse-mode subsystem typed against named-axis tensors. Phase 3 brings **WebGPU** acceleration ([ADR-0006](../docs/arc42/09-decisions/0006-adopt-webgpu-as-gpu-backend.md)) and a browser-runnable demo. Phase 4 publishes the **Jupyter Book site** built from this directory.

Each subsequent notebook is a focused chapter:

- `01_named-axes.ipynb` — formal grammar of label broadcast, error messages, the hybrid runtime/NTTP path.
- `02_function-and-reference-tensors.ipynb` — patterns where the 2016 ideas pay off in modern ML / optimisation contexts.
- `03_convolutions-as-inner-products.ipynb` — the centre piece of the original blog post, on the new API.
- `04_tex-bridge.ipynb` — the `_tex` parser internals; LyX export preview.
- `05_autograd-from-scratch.ipynb` — Phase 2.
- `06_webgpu-acceleration.ipynb` — Phase 3.
- `07_mlp-on-mnist.ipynb` — capstone.

If you read this far, you have effectively read the project's elevator pitch. Welcome.